# 05 - Champ de mouvement Lucas-Kanade

**Objectif de l'etape.** Estimer le champ de mouvement local sur les points de la voiture.

**Methode.** Des points caracteristiques sont detectes dans la ROI/mask de la voiture, puis Lucas-Kanade estime leurs deplacements entre deux frames. Les vecteurs dessines representent le champ de mouvement demande.

**Hypotheses.** Petits deplacements, illumination constante et coherence spatiale locale.

In [4]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATASET_DIR = PROJECT_ROOT / 'data' / 'car' / 'car-11' / 'img'
RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

image_files = sorted([p for p in DATASET_DIR.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}])
print('Nombre de frames:', len(image_files))

Nombre de frames: 1661


In [5]:
# Bbox manuelle de depart: ajuster si necessaire ou utiliser cv2.selectROI dans l'interface Tkinter.
# Format: x, y, w, h. Cette valeur ne vient pas d'annotations.
initial_bbox = (535, 300, 220, 105)
print('initial_bbox =', initial_bbox)

initial_bbox = (535, 300, 220, 105)


In [6]:
import cv2
from src.preprocessing import preprocess_image_with_method
from src.detection import detect_features_in_roi
from src.optical_flow import estimate_motion_field, draw_optical_flow_vectors
from src.visualization import draw_bbox, save_frame

frame1 = cv2.imread(str(image_files[0]))
frame2 = cv2.imread(str(image_files[1]))
gray1 = preprocess_image_with_method(frame1, method='stretching')['enhanced']
gray2 = preprocess_image_with_method(frame2, method='stretching')['enhanced']
points = detect_features_in_roi(gray1, initial_bbox)
field = estimate_motion_field(gray1, gray2, points)
print('Nombre de points suivis:', field['num_points'])
print('dx_global:', field['dx_global'], 'dy_global:', field['dy_global'])

vis = draw_optical_flow_vectors(frame2, field['good_old'], field['good_new'])
vis = draw_bbox(vis, initial_bbox)
save_frame(vis, RESULTS_DIR / 'optical_flow' / 'notebook_lk_vectors.png')

Nombre de points suivis: 80
dx_global: -0.8432464599609375 dy_global: -0.07639884948730469


True

**Resultats generes.** `results/optical_flow/notebook_lk_vectors.png` montre les vecteurs Lucas-Kanade sur la voiture.

**Interpretation.** Si les vecteurs sont coherents et orientes dans la meme direction, le champ de mouvement suit bien la voiture. Un faible nombre de points, des ombres, un manque de texture ou un deplacement trop grand peuvent produire des erreurs.